# Notebook 02: Preprocessing, Model Training, Evaluation & SHAP Analysis
### Predicting 30-Day Hospital Readmission in Patients With Diabetes
**Paper Reference:** Emi-Johnson & Nkrumah (*Cureus*, April 17, 2025, DOI: 10.7759/cureus.82437)

This notebook implements the complete machine learning workflow:
1. Preprocessing with `ColumnTransformer` (mean imputation, scaling, one-hot encoding).
2. Stratified 80/20 train/test splitting.
3. Training and evaluating 4 models: Logistic Regression, Random Forest, XGBoost, DNN.
4. Comprehensive test evaluation (Accuracy, Sensitivity, Precision, F1, ROC-AUC, PR-AUC).
5. SHAP interpretability on the top-performing XGBoost model.

In [ ]:
import sys
from pathlib import Path
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.preprocessing import split_and_preprocess_data
from src.evaluate_models import evaluate_all_models
from src.shap_analysis import run_shap_analysis
from src.config import RESULTS_DIR, SAVED_MODELS_DIR, PLOTS_DIR, SHAP_DIR

sns.set_theme(style='whitegrid')
print('Environment initialized successfully.')

## 1. Load Preprocessed Data and Trained Artifacts
We load the held-out test set ($N=20,354$) and the four models trained via cross-validation.

In [ ]:
data_dict = split_and_preprocess_data('../diabetic_data.csv')

models_dict = {
    'logistic_regression': joblib.load(SAVED_MODELS_DIR / 'logistic_regression.joblib'),
    'random_forest': joblib.load(SAVED_MODELS_DIR / 'random_forest.joblib'),
    'xgboost': joblib.load(SAVED_MODELS_DIR / 'xgboost.joblib'),
    'dnn': joblib.load(SAVED_MODELS_DIR / 'dnn.joblib'),
}
print('All four trained models successfully loaded from disk.')

## 2. Model Comparison and Evaluation Tables
Displaying overall metrics and comparing directly against the reference paper values.

In [ ]:
comparison_df = pd.read_csv(RESULTS_DIR / 'model_comparison.csv')
display_cols = ['Model', 'Accuracy', 'AUC_ROC', 'PR_AUC', 'Positive_Precision (PPV)', 'Positive_Recall (Sensitivity)', 'Positive_F1', 'Paper_AUC_ROC_Ref']
comparison_df[display_cols]

## 3. Class-Specific Breakdown and Misclassification Trends
Comparison of False Positives and False Negatives against Table 3 of Emi-Johnson & Nkrumah.

In [ ]:
misc_df = pd.read_csv(RESULTS_DIR / 'misclassification_summary.csv')
misc_df

## 4. Visualizing ROC and Precision-Recall Curves
Displaying the generated publication-grade performance curves.

In [ ]:
from PIL import Image
roc_img = Image.open(PLOTS_DIR / 'roc_curves.png')
plt.figure(figsize=(10, 7))
plt.imshow(roc_img)
plt.axis('off')
plt.title('Receiver Operating Characteristic Curves', fontsize=14)
plt.show()

pr_img = Image.open(PLOTS_DIR / 'precision_recall_curves.png')
plt.figure(figsize=(10, 7))
plt.imshow(pr_img)
plt.axis('off')
plt.title('Precision-Recall Curves', fontsize=14)
plt.show()

## 5. SHAP Interpretability Analysis (XGBoost)
Examining top features and directional impacts.

In [ ]:
shap_df = pd.read_csv(SHAP_DIR / 'shap_feature_importance.csv')
print('--- Top 15 Features by Mean Absolute SHAP Value ---')
print(shap_df.head(15))

shap_summary = Image.open(SHAP_DIR / 'shap_summary_plot.png')
plt.figure(figsize=(12, 9))
plt.imshow(shap_summary)
plt.axis('off')
plt.title('SHAP Summary Beeswarm Plot (XGBoost)', fontsize=14)
plt.show()